# 4.2 ACL Runtime 与资源

## 本节学习目标

- 理解 ACL 初始化和资源所有权
- 识别 RAII 清理顺序

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 生命周期

工程的 Runtime 构造函数调用 `aclInit -> aclrtSetDevice -> aclrtCreateContext -> aclrtSetCurrentContext -> aclrtCreateStream`。析构时先销毁 stream/context，再 `aclFinalize`。

## 内存与错误处理

DeviceBuffer 封装 `aclrtMalloc/aclrtFree`，copy helper 使用 `aclrtMemcpy`。宏将非成功状态转为异常，使失败不会静默继续。

## 查看 GEMM Runtime 封装

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

## 预期现象与结果分析

RAII 能覆盖正常退出和异常路径，但资源之间仍有依赖顺序。当前 Runtime 未显式 `aclrtResetDevice`，教学描述以真实源码为准。

## 课后实践

列出构造和析构顺序，并解释为何 stream 必须在 context/finalize 之前销毁。

参考答案见 `answer/04.02_answer.md`。